# Dataset 2 Recovery — 139 Failed Requests with Self-Contained Status Manager

**Version:** 1.3 (self-contained status/download/finalization workflow)

**Purpose:** recover exactly the 139 Dataset-2 requests that failed with
`credit_balance_exhausted`, then merge them with the 1,330 successful original
responses and recompute the final **1,469-sample** classification + topic results.

This notebook **does not rerun the 1,330 successful requests**.

Expected project layout:

```text
Depression_Thay Cach/
├── datasets/
├── runs/
│   └── d2_full_gpt-5-6-sol_sentiment_topics_v1_4/
│       ├── batch_input.jsonl
│       ├── batch_output.jsonl
│       ├── batch_errors.jsonl
│       ├── manifest.csv
│       ├── topic_codebook.csv
│       └── ...
└── D2_Recovery_139_GPT56_Sol_v1_0.ipynb
```

**Important:** your account balance was negative after the original run. Add enough
API credit to restore a positive balance before enabling the recovery submission.

Current GPT-5.6 Sol Batch pricing used only for a conservative planning guard:
short-context input $2.50/M, cached input $0.25/M, cache writes $3.125/M,
output $15/M. To avoid relying on cache savings, the local guard charges *all*
estimated input at the higher $3.125/M cache-write rate.

### Persistent API-key behavior

This revision stores the API key in the **operating system's secure credential store**
through Python's `keyring` package (for example, Windows Credential Manager on Windows).
The secret is **not written into the notebook, CSV/JSON files, or notebook outputs**.

- First run on this computer/user account: the notebook asks for the API key once,
  verifies it, and saves it in the OS credential store.
- Later kernel/Jupyter/computer restarts: the notebook loads the saved key automatically.
- If you move the notebook to another computer or another OS user account, you must enter it again.
- A `FORCE_REPLACE_SAVED_KEY` switch is included for deliberately replacing the stored credential.

Batch submission remains disabled by default.


**Key improvement in v1.3:** after the recovery Batch has been submitted, you do **not** need to rerun the notebook from the beginning after a kernel restart. Run only **Section 5** to check/download the Batch. After it reaches 139/139 successful, run only **Section 6** to merge and recompute the final D2 results.


## 1. Environment and paths


In [ ]:
from __future__ import annotations

import os, json, math, hashlib, getpass, subprocess, sys
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)



# ------------------------------------------------------------
# Persistent secret storage dependency
# ------------------------------------------------------------
# keyring uses the operating system's credential store.
INSTALL_KEYRING_IF_MISSING = True

try:
    import keyring
except ImportError:
    if not INSTALL_KEYRING_IF_MISSING:
        raise RuntimeError(
            "Python package 'keyring' is not installed. "
            "Set INSTALL_KEYRING_IF_MISSING=True and rerun this cell."
        )
    print("Installing 'keyring' for secure persistent API-key storage...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "keyring"])
    import keyring

print("Persistent credential support loaded.")

PROJECT_ROOT = Path.cwd()
ORIGINAL_RUN = PROJECT_ROOT / "runs" / "d2_full_gpt-5-6-sol_sentiment_topics_v1_4"
RECOVERY_RUN = PROJECT_ROOT / "runs" / "d2_recovery_139_gpt-5-6-sol_sentiment_topics_v1_4"
RECOVERY_RUN.mkdir(parents=True, exist_ok=True)

ORIGINAL_INPUT = ORIGINAL_RUN / "batch_input.jsonl"
ORIGINAL_OUTPUT = ORIGINAL_RUN / "batch_output.jsonl"
ORIGINAL_ERRORS = ORIGINAL_RUN / "batch_errors.jsonl"
ORIGINAL_MANIFEST = ORIGINAL_RUN / "manifest.csv"
TOPIC_CODEBOOK = ORIGINAL_RUN / "topic_codebook.csv"

required = [
    ORIGINAL_INPUT, ORIGINAL_OUTPUT, ORIGINAL_ERRORS,
    ORIGINAL_MANIFEST, TOPIC_CODEBOOK
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required original-run files:\n" + "\n".join(missing))

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

print("Project root:", PROJECT_ROOT)
print("Original D2 run:", ORIGINAL_RUN)
print("Recovery run:", RECOVERY_RUN)
print("Required original files: PASS")


## 2. Audit the failures and prepare **exactly 139 original requests** — $0.00

This cell reads the original OpenAI error file and requires every failed request to be:

- HTTP `429`
- error type `insufficient_quota`
- error code `credit_balance_exhausted`

It then selects the **exact original JSON request objects** from the original
`batch_input.jsonl`. No prompt, model, schema, text, topic taxonomy, or inference
parameter is changed.

The local recovery manifest is written as `UTF-8-SIG`, which displays Vietnamese
correctly when opened directly in Excel.


In [ ]:
# ---------- audit the 139 errors ----------
errors = [
    json.loads(line)
    for line in ORIGINAL_ERRORS.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

assert len(errors) == 139, f"Expected 139 errors, found {len(errors)}"

failed_ids = []
audit_rows = []

for e in errors:
    cid = e["custom_id"]
    response = e.get("response") or {}
    body = response.get("body") or {}
    err = body.get("error") or {}

    row = {
        "custom_id": cid,
        "status_code": response.get("status_code"),
        "error_type": err.get("type"),
        "error_code": err.get("code"),
        "error_message": err.get("message"),
    }
    audit_rows.append(row)
    failed_ids.append(cid)

audit = pd.DataFrame(audit_rows)

assert audit["custom_id"].nunique() == 139
assert set(audit["status_code"]) == {429}
assert set(audit["error_type"]) == {"insufficient_quota"}
assert set(audit["error_code"]) == {"credit_balance_exhausted"}

failed_id_set = set(failed_ids)

# ---------- select exact original API requests ----------
all_requests = [
    json.loads(line)
    for line in ORIGINAL_INPUT.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

assert len(all_requests) == 1469, f"Expected 1469 original requests, found {len(all_requests)}"

recovery_requests = [r for r in all_requests if r["custom_id"] in failed_id_set]

assert len(recovery_requests) == 139
assert len({r["custom_id"] for r in recovery_requests}) == 139
assert {r["custom_id"] for r in recovery_requests} == failed_id_set

recovery_input = RECOVERY_RUN / "batch_input_recovery_139.jsonl"

with recovery_input.open("w", encoding="utf-8") as f:
    for r in recovery_requests:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# ---------- local gold-label manifest for the same 139 ----------
manifest = pd.read_csv(ORIGINAL_MANIFEST)
assert len(manifest) == 1469
assert manifest["custom_id"].nunique() == 1469

recovery_manifest = manifest[manifest["custom_id"].isin(failed_id_set)].copy()
assert len(recovery_manifest) == 139
assert set(recovery_manifest["custom_id"]) == failed_id_set

recovery_manifest.to_csv(
    RECOVERY_RUN / "manifest_failed_139.csv",
    index=False,
    encoding="utf-8-sig",
)
audit.to_csv(
    RECOVERY_RUN / "failure_audit_139.csv",
    index=False,
    encoding="utf-8-sig",
)

# ---------- conservative cost guard ----------
# Current short-context Batch rates checked 2026-09-12.
# To be conservative, all estimated input is charged at the higher cache-write rate.
INPUT_GUARD_RATE_PER_MTOK = 3.125
OUTPUT_RATE_PER_MTOK = 15.00
CHARS_PER_TOKEN_GUARD = 2.0
REQUEST_OVERHEAD_TOKENS = 96

estimated_input_tokens = 0
max_output_tokens_total = 0

for r in recovery_requests:
    body_text = json.dumps(r["body"], ensure_ascii=False, separators=(",", ":"))
    estimated_input_tokens += (
        math.ceil(len(body_text) / CHARS_PER_TOKEN_GUARD)
        + REQUEST_OVERHEAD_TOKENS
    )
    max_output_tokens_total += int(r["body"].get("max_output_tokens", 48))

estimated_guard_usd = (
    estimated_input_tokens / 1_000_000 * INPUT_GUARD_RATE_PER_MTOK
    + max_output_tokens_total / 1_000_000 * OUTPUT_RATE_PER_MTOK
)

MAX_RECOVERY_REQUESTS = 139
MAX_RECOVERY_ESTIMATE_USD = 1.00

assert len(recovery_requests) == MAX_RECOVERY_REQUESTS
assert estimated_guard_usd <= MAX_RECOVERY_ESTIMATE_USD

recovery_config = {
    "created_at_utc": utc_now(),
    "purpose": "recover exactly 139 D2 requests failed due to exhausted credit",
    "original_run": str(ORIGINAL_RUN.resolve()),
    "n_requests": 139,
    "failed_ids_sha256": hashlib.sha256(
        "\n".join(sorted(failed_id_set)).encode("utf-8")
    ).hexdigest(),
    "recovery_input_sha256": sha256_file(recovery_input),
    "pricing_checked_date": "2026-09-12",
    "cost_guard": {
        "input_rate_used_per_mtok": INPUT_GUARD_RATE_PER_MTOK,
        "output_rate_per_mtok": OUTPUT_RATE_PER_MTOK,
        "estimated_input_tokens": estimated_input_tokens,
        "output_tokens_upper_bound": max_output_tokens_total,
        "estimated_guard_usd": estimated_guard_usd,
        "max_allowed_usd": MAX_RECOVERY_ESTIMATE_USD,
    },
}
write_json(RECOVERY_RUN / "recovery_config.json", recovery_config)

print("RECOVERY PREPARATION: PASS")
print("Failed requests:", len(recovery_requests))
print("Gold labels:", recovery_manifest["gold_label"].value_counts().sort_index().to_dict())
print("Recovery JSONL:", recovery_input)
print("Recovery JSONL SHA256:", sha256_file(recovery_input))
print(f"Conservative recovery cost guard estimate: ${estimated_guard_usd:.4f}")
print(f"Hard local estimate cap: ${MAX_RECOVERY_ESTIMATE_USD:.2f}")
print("Nothing has been submitted by this cell.")


## 3. Load the API key permanently from the OS credential store + safe submission gate

This notebook now saves the API key **persistently but not inside the notebook**.

It uses the operating system's credential manager through the Python `keyring` package.
On Windows this normally maps to **Windows Credential Manager**; on other supported systems
it uses the corresponding OS keychain/credential backend.

### First run

The next cell checks whether a saved OpenAI API key already exists for this project.

- If no saved key exists, a hidden prompt appears:
  ```text
  Enter OpenAI API key to SAVE securely on this computer (hidden):
  ```
- Paste the key and press Enter.
- The notebook first verifies the key with OpenAI.
- Only after authentication passes does it save the key in the OS credential store.

The key is never printed and is never written to the notebook.

### Future experiments / restarts

On later runs, the notebook automatically retrieves the saved key and sets:

```python
os.environ["OPENAI_API_KEY"]
```

so other experiment notebooks can use the same credential during that Python session.

### Replacing the saved key

If you later want to replace it with a different OpenAI account/key, set:

```python
FORCE_REPLACE_SAVED_KEY = True
```

for one run. The notebook will prompt for the replacement key, verify it, and overwrite
the saved credential. Then set it back to `False`.

### Submission safety

The first `Run All` is still safe:

```python
ALLOW_RECOVERY_SUBMISSION = False
CONFIRM_RECOVERY_SUBMISSION = "NO"
```

After the local recovery preparation and API-key authentication pass, authorize only the
139-request recovery by changing:

```python
ALLOW_RECOVERY_SUBMISSION = True
CONFIRM_RECOVERY_SUBMISSION = "SUBMIT_D2_RECOVERY_139"
```

Then run Section 4 once.


In [ ]:
# ============================================================
# PERSISTENT OPENAI API KEY — OS CREDENTIAL STORE
# ============================================================

# These names identify the secret inside the OS credential manager.
# They are not secret.
KEYRING_SERVICE = "OpenAI API - Vietnamese Depression Project"
KEYRING_USERNAME = "openai_api_key"

# Set True only when you intentionally want to replace the saved API key.
FORCE_REPLACE_SAVED_KEY = False

# Batch submission stays OFF during the initial Run All.
ALLOW_RECOVERY_SUBMISSION = False
CONFIRM_RECOVERY_SUBMISSION = "NO"
REQUIRED_CONFIRMATION = "SUBMIT_D2_RECOVERY_139"


def verify_openai_key(candidate_key: str) -> None:
    """Verify a candidate key without printing it or submitting model inference."""
    from openai import OpenAI

    client = OpenAI(api_key=candidate_key)
    # Lightweight authenticated request; does not run the 139 recovery items.
    client.models.list()


def save_verified_key_to_os(candidate_key: str) -> None:
    """Save only after authentication has succeeded."""
    keyring.set_password(KEYRING_SERVICE, KEYRING_USERNAME, candidate_key)


# Check the keyring backend before asking for a secret.
backend = keyring.get_keyring()
backend_name = f"{backend.__class__.__module__}.{backend.__class__.__name__}"
backend_priority = getattr(backend, "priority", 0)

if not backend_priority or backend_priority <= 0:
    raise RuntimeError(
        "No usable secure OS keyring backend was detected. "
        "For safety, this notebook will NOT save the API key in plaintext. "
        "Install/configure a supported keyring backend and rerun this cell."
    )

print("Credential backend:", backend_name)

saved_key = None if FORCE_REPLACE_SAVED_KEY else keyring.get_password(
    KEYRING_SERVICE, KEYRING_USERNAME
)

if saved_key:
    # Validate the stored credential before putting it into the environment.
    try:
        verify_openai_key(saved_key)
    except Exception as exc:
        # Remove invalid/stale stored credential and require a fresh key.
        try:
            keyring.delete_password(KEYRING_SERVICE, KEYRING_USERNAME)
        except Exception:
            pass
        saved_key = None
        os.environ.pop("OPENAI_API_KEY", None)
        print("Previously saved API key was invalid or unavailable and has been removed.")

if not saved_key:
    candidate_key = getpass.getpass(
        "Enter OpenAI API key to SAVE securely on this computer (hidden): "
    ).strip()

    if not candidate_key:
        os.environ.pop("OPENAI_API_KEY", None)
        raise RuntimeError("No API key entered. Nothing was saved.")

    try:
        verify_openai_key(candidate_key)
    except Exception as exc:
        # Do not save a key that failed authentication.
        candidate_key = None
        os.environ.pop("OPENAI_API_KEY", None)
        raise RuntimeError(
            "API KEY AUTHENTICATION FAILED. The key was NOT saved. "
            "Check that you copied the correct key and that the API project/account is active."
        ) from exc

    save_verified_key_to_os(candidate_key)
    saved_key = candidate_key
    candidate_key = None

    print("API key authentication: PASS")
    print("API key saved securely in the operating system credential store.")
else:
    print("Saved API key authentication: PASS")
    print("Loaded API key from the operating system credential store.")

# Load the verified key into this Python/Jupyter session.
os.environ["OPENAI_API_KEY"] = saved_key

# Drop the local reference; the environment variable remains for API clients in this session.
saved_key = None

print()
print("OPENAI_API_KEY available in this session:", bool(os.environ.get("OPENAI_API_KEY")))
print("Recovery submission enabled?", ALLOW_RECOVERY_SUBMISSION)
print("Required confirmation:", REQUIRED_CONFIRMATION)
print("No recovery Batch has been submitted by this cell.")


## 4. Submit the 139-request recovery Batch — run once only after authorization

Before running this paid submission cell:

- Section 2 must report `RECOVERY PREPARATION: PASS`;
- Section 3 must report either:
  - `API key authentication: PASS`, or
  - `Saved API key authentication: PASS`;
- your API account must have a positive credit balance;
- change the Section-3 gate to:

```python
ALLOW_RECOVERY_SUBMISSION = True
CONFIRM_RECOVERY_SUBMISSION = "SUBMIT_D2_RECOVERY_139"
```

This cell refuses duplicate submission if `recovery_batch_state.json` already exists.
It submits **only the 139 previously failed Dataset-2 requests**.


In [ ]:
RECOVERY_STATE = RECOVERY_RUN / "recovery_batch_state.json"
RECOVERY_UPLOAD_STATE = RECOVERY_RUN / "recovery_upload_state.json"
RECOVERY_INPUT = RECOVERY_RUN / "batch_input_recovery_139.jsonl"

def submit_recovery():
    if not ALLOW_RECOVERY_SUBMISSION:
        print("SAFE STOP: recovery submission is disabled. Nothing uploaded or submitted.")
        return None

    if CONFIRM_RECOVERY_SUBMISSION != REQUIRED_CONFIRMATION:
        raise RuntimeError(
            f"Confirmation mismatch. Required exactly: {REQUIRED_CONFIRMATION}"
        )

    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError(
            "OPENAI_API_KEY is not available. Run Section 3 to load the verified saved key first."
        )

    if RECOVERY_STATE.exists():
        state = read_json(RECOVERY_STATE)
        print("SKIP: recovery Batch was already created:", state.get("batch_id"))
        print("Last saved status:", state.get("status"))
        return state.get("batch_id")

    cfg = read_json(RECOVERY_RUN / "recovery_config.json")
    if sha256_file(RECOVERY_INPUT) != cfg["recovery_input_sha256"]:
        raise RuntimeError("Recovery JSONL changed after preparation. Stop.")

    if int(cfg["n_requests"]) != 139:
        raise RuntimeError("Recovery request count is not exactly 139. Stop.")

    if float(cfg["cost_guard"]["estimated_guard_usd"]) > 1.00:
        raise RuntimeError("Recovery local cost guard exceeded $1.00. Stop.")

    from openai import OpenAI
    client = OpenAI()

    # Reuse uploaded file if upload succeeded but Batch creation failed.
    if RECOVERY_UPLOAD_STATE.exists():
        upload_state = read_json(RECOVERY_UPLOAD_STATE)
        input_file_id = upload_state["input_file_id"]
        print("Reusing previously uploaded file:", input_file_id)
    else:
        with RECOVERY_INPUT.open("rb") as f:
            uploaded = client.files.create(file=f, purpose="batch")
        input_file_id = uploaded.id
        write_json(RECOVERY_UPLOAD_STATE, {
            "uploaded_at_utc": utc_now(),
            "input_file_id": input_file_id,
            "recovery_input_sha256": cfg["recovery_input_sha256"],
        })
        print("Uploaded recovery input:", input_file_id)

    batch = client.batches.create(
        input_file_id=input_file_id,
        endpoint="/v1/responses",
        completion_window="24h",
        metadata={
            "project": "vietnamese_depression_signals",
            "experiment": "d2_recovery_139_sentiment_topics_v1_4",
            "dataset": "2",
            "mode": "recovery_139",
            "run_name": RECOVERY_RUN.name,
        },
    )

    state = {
        "submitted_at_utc": utc_now(),
        "last_checked_at_utc": utc_now(),
        "batch_id": batch.id,
        "input_file_id": input_file_id,
        "status": getattr(batch, "status", None),
        "output_file_id": getattr(batch, "output_file_id", None),
        "error_file_id": getattr(batch, "error_file_id", None),
        "request_count": 139,
    }
    write_json(RECOVERY_STATE, state)

    print("RECOVERY BATCH SUBMITTED")
    print("Batch ID:", batch.id)
    print("Initial status:", getattr(batch, "status", None))
    return batch.id

recovery_batch_id = submit_recovery()


## ⭐ 5. Self-contained recovery manager — run **only this cell** later

This cell is deliberately independent of Sections 1–4.

After the Batch has already been submitted, even if you close Jupyter or restart the kernel,
you can reopen this notebook and run **only this cell**.

It will automatically:

1. locate the project and recovery folder;
2. load the saved API key from the OS credential store;
3. read the saved `recovery_batch_state.json`;
4. retrieve the Batch status from OpenAI;
5. show `completed / total / failed`;
6. download the output/error file when the Batch is complete.

**It never creates or submits a Batch.**


In [ ]:
# ============================================================
# ⭐ SELF-CONTAINED RECOVERY MANAGER
# Safe to run by itself after a kernel/Jupyter restart.
# NEVER creates or submits a Batch.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import os, json, subprocess, sys

# ---------- find project ----------
def _find_project_root():
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "runs" / "d2_recovery_139_gpt-5-6-sol_sentiment_topics_v1_4").exists():
            return p
        if (p / "runs").is_dir() and (p / "datasets").is_dir():
            return p
    return start

PROJECT_ROOT_MGR = _find_project_root()
RECOVERY_RUN_MGR = (
    PROJECT_ROOT_MGR
    / "runs"
    / "d2_recovery_139_gpt-5-6-sol_sentiment_topics_v1_4"
)
RECOVERY_STATE_MGR = RECOVERY_RUN_MGR / "recovery_batch_state.json"

print("Project root:", PROJECT_ROOT_MGR)
print("Recovery folder:", RECOVERY_RUN_MGR)

if not RECOVERY_STATE_MGR.exists():
    raise FileNotFoundError(
        "recovery_batch_state.json was not found. "
        "This usually means the recovery Batch was not submitted from this project folder."
    )

# ---------- load persistent API key from OS credential store ----------
try:
    import keyring
except ImportError:
    print("Installing 'keyring' so the saved API key can be loaded...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "keyring"])
    import keyring

KEYRING_SERVICE = "OpenAI API - Vietnamese Depression Project"
KEYRING_USERNAME = "openai_api_key"

saved_key = keyring.get_password(KEYRING_SERVICE, KEYRING_USERNAME)
if not saved_key:
    raise RuntimeError(
        "No saved OpenAI API key was found in the OS credential store. "
        "Run Section 3 once to save the key, then use only this manager cell afterward."
    )

os.environ["OPENAI_API_KEY"] = saved_key
saved_key = None

# ---------- utilities ----------
def _read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _write_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

def _utc_now():
    return datetime.now(timezone.utc).isoformat()

def _download_text(client, file_id):
    resp = client.files.content(file_id)

    text = getattr(resp, "text", None)
    if isinstance(text, str):
        return text

    content = getattr(resp, "content", None)
    if isinstance(content, (bytes, bytearray)):
        return bytes(content).decode("utf-8")

    if hasattr(resp, "read"):
        data = resp.read()
        return data.decode("utf-8") if isinstance(data, bytes) else str(data)

    return str(resp)

# ---------- retrieve status ----------
from openai import OpenAI
client_mgr = OpenAI()

state = _read_json(RECOVERY_STATE_MGR)
batch_id = state.get("batch_id")
if not batch_id:
    raise RuntimeError("Saved recovery state does not contain a batch_id.")

batch = client_mgr.batches.retrieve(batch_id)

counts = getattr(batch, "request_counts", None)
total = getattr(counts, "total", None) if counts else None
completed = getattr(counts, "completed", None) if counts else None
failed = getattr(counts, "failed", None) if counts else None

state.update({
    "last_checked_at_utc": _utc_now(),
    "status": getattr(batch, "status", None),
    "output_file_id": getattr(batch, "output_file_id", None),
    "error_file_id": getattr(batch, "error_file_id", None),
    "request_total": total,
    "request_completed": completed,
    "request_failed": failed,
})
_write_json(RECOVERY_STATE_MGR, state)

print("\nBatch ID:", batch_id)
print("Status:", state["status"])
print("Requests:", completed, "/", total, "completed;", failed, "failed")

# ---------- download when complete ----------
if state["status"] == "completed":
    output_path = RECOVERY_RUN_MGR / "recovery_batch_output.jsonl"
    error_path = RECOVERY_RUN_MGR / "recovery_batch_errors.jsonl"

    if state.get("output_file_id"):
        if not output_path.exists():
            output_path.write_text(
                _download_text(client_mgr, state["output_file_id"]),
                encoding="utf-8",
            )
            print("Downloaded output:", output_path)
        else:
            print("Output already downloaded:", output_path)

    if state.get("error_file_id"):
        if not error_path.exists():
            error_path.write_text(
                _download_text(client_mgr, state["error_file_id"]),
                encoding="utf-8",
            )
            print("Downloaded error file:", error_path)
        else:
            print("Error file already downloaded:", error_path)

    if completed == 139 and failed == 0:
        print("\nRECOVERY COMPLETE: all 139 requests succeeded.")
        print("Next step: run ONLY Section 6.")
    else:
        print(
            "\nRecovery Batch completed, but not all 139 requests succeeded. "
            "Do NOT merge yet; inspect the downloaded error file."
        )
else:
    print("\nThe recovery Batch is not finished yet.")
    print("Later, rerun ONLY this Section 5 cell.")


## 6. Self-contained final merge/evaluation — run **only this cell** after 139/139 success

This cell is also independent of Sections 1–5.

Once Section 5 reports:

```text
RECOVERY COMPLETE: all 139 requests succeeded.
```

run **only this Section 6 cell**. It loads the original 1,330 successes and the 139 recovered
successes directly from disk, verifies all 1,469 `custom_id` values, then writes the final
D2 predictions, classification metrics, bootstrap confidence intervals, and topic tables.

It never calls GPT and never submits an API request.


In [ ]:
# ============================================================
# SELF-CONTAINED FINAL D2 MERGE + EVALUATION
# No API inference and no Batch submission.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import os, json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)

# ---------- find project ----------
def _find_project_root_final():
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "runs").is_dir() and (p / "datasets").is_dir():
            return p
    return start

PROJECT_ROOT_FINAL = _find_project_root_final()

ORIGINAL_RUN_FINAL = (
    PROJECT_ROOT_FINAL
    / "runs"
    / "d2_full_gpt-5-6-sol_sentiment_topics_v1_4"
)
RECOVERY_RUN_FINAL = (
    PROJECT_ROOT_FINAL
    / "runs"
    / "d2_recovery_139_gpt-5-6-sol_sentiment_topics_v1_4"
)

ORIGINAL_OUTPUT_FINAL = ORIGINAL_RUN_FINAL / "batch_output.jsonl"
ORIGINAL_MANIFEST_FINAL = ORIGINAL_RUN_FINAL / "manifest.csv"
TOPIC_CODEBOOK_FINAL = ORIGINAL_RUN_FINAL / "topic_codebook.csv"
RECOVERY_OUTPUT_FINAL = RECOVERY_RUN_FINAL / "recovery_batch_output.jsonl"
RECOVERY_STATE_FINAL = RECOVERY_RUN_FINAL / "recovery_batch_state.json"

required = [
    ORIGINAL_OUTPUT_FINAL,
    ORIGINAL_MANIFEST_FINAL,
    TOPIC_CODEBOOK_FINAL,
    RECOVERY_OUTPUT_FINAL,
    RECOVERY_STATE_FINAL,
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Required file(s) missing. Run Section 5 after the recovery Batch finishes:\n"
        + "\n".join(missing)
    )

def _read_json_final(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _write_json_final(path, obj):
    Path(path).write_text(
        json.dumps(obj, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

def _utc_now_final():
    return datetime.now(timezone.utc).isoformat()

# Require a completed 139/139 recovery before merging.
state = _read_json_final(RECOVERY_STATE_FINAL)
if not (
    state.get("status") == "completed"
    and int(state.get("request_completed") or 0) == 139
    and int(state.get("request_failed") or 0) == 0
):
    raise RuntimeError(
        "Recovery is not recorded as 139/139 completed with 0 failures. "
        "Run Section 5 first; do not merge incomplete results."
    )

# ---------- load 1,330 original + 139 recovered ----------
original_objs = [
    json.loads(line)
    for line in ORIGINAL_OUTPUT_FINAL.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
recovery_objs = [
    json.loads(line)
    for line in RECOVERY_OUTPUT_FINAL.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

assert len(original_objs) == 1330, (
    f"Expected 1330 original successes, got {len(original_objs)}"
)
assert len(recovery_objs) == 139, (
    f"Expected 139 recovery successes, got {len(recovery_objs)}"
)

by_id = {}

for obj in original_objs + recovery_objs:
    cid = obj["custom_id"]
    status_code = (obj.get("response") or {}).get("status_code")

    if status_code != 200:
        raise RuntimeError(
            f"Non-200 response in merged success set: {cid}, status={status_code}"
        )
    if cid in by_id:
        raise RuntimeError(f"Duplicate custom_id while merging: {cid}")

    by_id[cid] = obj

manifest = pd.read_csv(ORIGINAL_MANIFEST_FINAL)
assert len(manifest) == 1469
expected_ids = set(manifest["custom_id"].astype(str))

assert len(by_id) == 1469
assert set(by_id) == expected_ids

# Save a reproducible merged raw output in manifest order.
merged_jsonl = RECOVERY_RUN_FINAL / "final_1469_batch_output_merged.jsonl"

with merged_jsonl.open("w", encoding="utf-8") as f:
    for cid in manifest["custom_id"].astype(str):
        f.write(json.dumps(by_id[cid], ensure_ascii=False) + "\n")

# ---------- topic codebook ----------
codebook = pd.read_csv(TOPIC_CODEBOOK_FINAL)
topic_name = dict(zip(codebook["topic_id"], codebook["topic_name"]))

valid_primary = set(topic_name)
valid_secondary = (valid_primary - {"T00"}) | {"NONE"}

def _extract_output_text_final(body):
    direct = body.get("output_text")
    if isinstance(direct, str) and direct.strip():
        return direct.strip()

    pieces = []
    for item in body.get("output", []) or []:
        if not isinstance(item, dict):
            continue
        for content in item.get("content", []) or []:
            if isinstance(content, dict) and content.get("type") == "output_text":
                txt = content.get("text")
                if isinstance(txt, str):
                    pieces.append(txt)

    return "".join(pieces).strip()

manifest_by_id = manifest.set_index("custom_id")

rows = []
total_input = 0
total_output = 0
total_cached = 0
total_cache_write = 0

for cid in manifest["custom_id"].astype(str):
    obj = by_id[cid]
    body = (obj.get("response") or {}).get("body") or {}
    raw = _extract_output_text_final(body)

    try:
        ann = json.loads(raw)
    except Exception as exc:
        raise RuntimeError(f"Invalid structured JSON for {cid}: {raw!r}") from exc

    expected_keys = {
        "predicted_label",
        "primary_topic",
        "secondary_topic",
    }
    if set(ann) != expected_keys:
        raise RuntimeError(f"Unexpected output keys for {cid}: {ann}")

    label = ann["predicted_label"]
    primary = ann["primary_topic"]
    secondary = ann["secondary_topic"]

    if label not in (0, 1):
        raise RuntimeError(f"Invalid predicted_label for {cid}: {label}")
    if primary not in valid_primary:
        raise RuntimeError(f"Invalid primary_topic for {cid}: {primary}")
    if secondary not in valid_secondary:
        raise RuntimeError(f"Invalid secondary_topic for {cid}: {secondary}")
    if primary == "T00" and secondary != "NONE":
        raise RuntimeError(f"T00 requires secondary_topic=NONE for {cid}")
    if secondary != "NONE" and secondary == primary:
        raise RuntimeError(f"Duplicate primary/secondary topic for {cid}")

    m = manifest_by_id.loc[cid]

    usage = body.get("usage") or {}
    details = usage.get("input_tokens_details") or {}

    input_tokens = int(usage.get("input_tokens") or 0)
    output_tokens = int(usage.get("output_tokens") or 0)
    cached_tokens = int(details.get("cached_tokens") or 0)
    cache_write_tokens = int(details.get("cache_write_tokens") or 0)

    total_input += input_tokens
    total_output += output_tokens
    total_cached += cached_tokens
    total_cache_write += cache_write_tokens

    rows.append({
        "custom_id": cid,
        "dataset": 2,
        "source_row_id": int(m["source_row_id"]),
        "gold_label": int(m["gold_label"]),
        "predicted_label": int(label),
        "primary_topic": primary,
        "primary_topic_name": topic_name.get(primary, ""),
        "secondary_topic": secondary,
        "secondary_topic_name": (
            "NONE" if secondary == "NONE" else topic_name.get(secondary, "")
        ),
        "input_tokens": input_tokens,
        "cached_input_tokens": cached_tokens,
        "cache_write_tokens": cache_write_tokens,
        "output_tokens": output_tokens,
        "text_vi": m["text_vi"],
    })

final = pd.DataFrame(rows)

# UTF-8-SIG displays Vietnamese correctly in Excel.
final_predictions = RECOVERY_RUN_FINAL / "final_1469_predictions.csv"
final.to_csv(final_predictions, index=False, encoding="utf-8-sig")

# ---------- classification metrics ----------
y_true = final["gold_label"].astype(int).to_numpy()
y_pred = final["predicted_label"].astype(int).to_numpy()

def _bootstrap_ci_final(metric_fn, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        vals.append(metric_fn(y_true[idx], y_pred[idx]))

    lo, hi = np.percentile(vals, [2.5, 97.5])
    return [float(lo), float(hi)]

metrics = {
    "dataset": 2,
    "n": 1469,
    "recovery": {
        "original_successes": 1330,
        "recovered_successes": 139,
        "final_missing": 0,
    },
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "precision_class1": float(
        precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    ),
    "recall_class1": float(
        recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    ),
    "f1_class1": float(
        f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    ),
    "macro_f1": float(
        f1_score(y_true, y_pred, average="macro", zero_division=0)
    ),
    "confusion_matrix_labels_0_1": confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).tolist(),
    "classification_report": classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        output_dict=True,
        zero_division=0,
    ),
    "bootstrap_95pct_ci": {
        "accuracy": _bootstrap_ci_final(accuracy_score),
        "f1_class1": _bootstrap_ci_final(
            lambda yt, yp: f1_score(
                yt, yp, pos_label=1, zero_division=0
            )
        ),
        "macro_f1": _bootstrap_ci_final(
            lambda yt, yp: f1_score(
                yt, yp, average="macro", zero_division=0
            )
        ),
    },
    "usage": {
        "input_tokens": int(total_input),
        "output_tokens": int(total_output),
        "cached_input_tokens_reported": int(total_cached),
        "cache_write_tokens_reported": int(total_cache_write),
    },
    "evaluated_at_utc": _utc_now_final(),
    "complete_for_final_reporting": True,
}

final_metrics = RECOVERY_RUN_FINAL / "final_1469_metrics.json"
_write_json_final(final_metrics, metrics)

# ---------- topic tables ----------
topic_order = ["T00"] + [f"T{i:02d}" for i in range(1, 13)]

primary_dist = (
    final["primary_topic"]
    .value_counts()
    .reindex(topic_order, fill_value=0)
    .rename_axis("topic_id")
    .reset_index(name="count")
)
primary_dist["topic_name"] = primary_dist["topic_id"].map(topic_name)
primary_dist["percent"] = primary_dist["count"] / len(final) * 100.0
primary_dist.to_csv(
    RECOVERY_RUN_FINAL / "final_1469_topic_primary_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)

secondary_present = final[
    final["secondary_topic"].isin([f"T{i:02d}" for i in range(1, 13)])
]

secondary_dist = (
    secondary_present["secondary_topic"]
    .value_counts()
    .reindex([f"T{i:02d}" for i in range(1, 13)], fill_value=0)
    .rename_axis("topic_id")
    .reset_index(name="count")
)
secondary_dist["topic_name"] = secondary_dist["topic_id"].map(topic_name)
secondary_dist["percent_of_all_texts"] = (
    secondary_dist["count"] / len(final) * 100.0
)
secondary_dist.to_csv(
    RECOVERY_RUN_FINAL / "final_1469_topic_secondary_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)

topic_by_gold_counts = pd.crosstab(
    final["primary_topic"], final["gold_label"], dropna=False
).reindex(topic_order, fill_value=0)
topic_by_gold_counts.index.name = "topic_id"
topic_by_gold_counts.to_csv(
    RECOVERY_RUN_FINAL / "final_1469_topic_by_gold_label_counts.csv",
    encoding="utf-8-sig",
)

topic_by_gold_pct = (
    pd.crosstab(
        final["primary_topic"],
        final["gold_label"],
        normalize="columns",
        dropna=False,
    )
    .reindex(topic_order, fill_value=0)
    * 100.0
)
topic_by_gold_pct.index.name = "topic_id"
topic_by_gold_pct.to_csv(
    RECOVERY_RUN_FINAL
    / "final_1469_topic_by_gold_label_percent_within_label.csv",
    encoding="utf-8-sig",
)

pairs = (
    final[final["secondary_topic"] != "NONE"]
    .groupby(["primary_topic", "secondary_topic"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
pairs.to_csv(
    RECOVERY_RUN_FINAL / "final_1469_topic_primary_secondary_pairs.csv",
    index=False,
    encoding="utf-8-sig",
)

print("FINAL D2 MERGE/EVALUATION: PASS")
print("Final rows:", len(final))
print("Missing:", 1469 - len(final))
print("Accuracy:", f"{metrics['accuracy']:.4f}")
print("Precision class 1:", f"{metrics['precision_class1']:.4f}")
print("Recall class 1:", f"{metrics['recall_class1']:.4f}")
print("F1 class 1:", f"{metrics['f1_class1']:.4f}")
print("Macro-F1:", f"{metrics['macro_f1']:.4f}")
print("Confusion matrix [0,1]:", metrics["confusion_matrix_labels_0_1"])
print("\nSaved:")
print(" ", final_predictions)
print(" ", final_metrics)
print(" ", RECOVERY_RUN_FINAL / "final_1469_topic_primary_distribution.csv")
print(" ", RECOVERY_RUN_FINAL / "final_1469_topic_secondary_distribution.csv")
print(" ", RECOVERY_RUN_FINAL / "final_1469_topic_by_gold_label_counts.csv")
print(" ", RECOVERY_RUN_FINAL / "final_1469_topic_by_gold_label_percent_within_label.csv")
print(" ", RECOVERY_RUN_FINAL / "final_1469_topic_primary_secondary_pairs.csv")
